## RAG Implementation using OpenSearch and KServe

In [57]:
import requests

In [59]:
import os

Read environment variables

In [3]:
os_endpoint = f"https://{os.environ['OPENSEARCH_ENDPOINTS']}"
os_username=f"{os.environ['OPENSEARCH_USERNAME']}"
os_password=f"{os.environ['OPENSEARCH_PASSWORD']}"
os_certificate=f"{os.environ['OPENSEARCH_CA_CERT_PATH']}"

In [4]:
os_endpoint

'https://10.254.246.126:9200,10.254.246.214:9200,10.254.246.81:9200'

In [5]:
os_username

'opensearch-client_5'

In [6]:
os_certificate

'/etc/data-kubeflow-integrator/opensearch_ca.crt'

In [98]:
base_url = f"{os_endpoint.split(',')[0].rstrip('/')}"

print(base_url)

def make_request(url, method, payload):
    _method = method.lower()
    if _method == "post":
        f = requests.post
    elif _method == "get":
        f = requests.get
    elif _method == "put":
        f = requests.put
    else:
        raise ValueError(f"method {method} not supported")

    _url = f"{base_url}/{url}"

    headers = {
        "Content-Type": "application/json"
    }
    
    params = {
        "auth": (os_username, os_password),
        "headers": headers,
        "verify": os_certificate
    } | ({"json": payload} if payload else {})

    response = f(
        _url, **params
    )
    # 5. Print the results
    print(f"Status Code: {response.status_code}")
    
    json_response = response.json()
    print("Response JSON:", json_response)
    
    return json_response

https://10.254.246.126:9200


## Create Indexed Knowledge base of Documents

Provide the name of the index collection

In [113]:
index_name = "knn-faiss-avx2-index1"

### Register and Deploy Embedding Model

In [68]:
model_group_id = make_request(
    "_plugins/_ml/model_groups/_register",
    "POST",
    {
        "name": "rag-model-group-3",
        "description": "A model group for RAG use case models"
    }
)["model_group_id"]

Status Code: 200
Response JSON: {'model_group_id': 'a-3JW54BnAiy9mby28Jn', 'status': 'CREATED'}


In [70]:
model_group_id

'a-3JW54BnAiy9mby28Jn'

In [69]:
task_id = make_request(
    "_plugins/_ml/models/_register",
    "POST",
    {
        "name": "huggingface/sentence-transformers/msmarco-distilbert-base-tas-b",
        "version": "1.0.2",
        "model_group_id": model_group_id,
        "model_format": "TORCH_SCRIPT"
    }
)["task_id"]

Status Code: 200
Response JSON: {'task_id': 'bO3JW54BnAiy9mby68Jp', 'status': 'CREATED'}


In [71]:
task_id

'bO3JW54BnAiy9mby68Jp'

In [72]:
from tenacity import retry, stop_after_attempt, wait_exponential

In [81]:
@retry(
    wait=wait_exponential(multiplier=2, min=1, max=10),
    stop=stop_after_attempt(10),
    reraise=True,
)
def get_model_id(task_id):
    json_response = make_request(
        f"_plugins/_ml/tasks/{task_id}",
        "GET",
        {}
    )
    assert json_response["state"] == "COMPLETED"
    return json_response["model_id"]

In [82]:
model_id = get_model_id(task_id)

Status Code: 200
Response JSON: {'model_id': 'OaPJW54B8MAT2xzP9bby', 'task_type': 'REGISTER_MODEL', 'function_name': 'TEXT_EMBEDDING', 'state': 'COMPLETED', 'worker_node': ['RZVRRu2TTiGOj16wknbgCQ'], 'create_time': 1779656420199, 'last_update_time': 1779656460685, 'is_async': True}


In [83]:
model_id

'OaPJW54B8MAT2xzP9bby'

In [86]:
make_request(
    f"_plugins/_ml/models/{model_id}/_deploy",
    "POST",
    {}
)

Status Code: 200
Response JSON: {'task_id': 'be3OW54BnAiy9mbyZsLr', 'task_type': 'DEPLOY_MODEL', 'status': 'CREATED'}


{'task_id': 'be3OW54BnAiy9mbyZsLr',
 'task_type': 'DEPLOY_MODEL',
 'status': 'CREATED'}

Test the embedding model

In [88]:
inference_output = make_request(
    f"_plugins/_ml/_predict/text_embedding/{model_id}",
    "POST",
    {
        "text_docs":[ "What is Ubuntu Pro?"],
        "return_number": "true",
        "target_response": ["sentence_embedding"]
    }
)

Status Code: 200
Response JSON: {'inference_results': [{'output': [{'name': 'sentence_embedding', 'data_type': 'FLOAT32', 'shape': [768], 'data': [0.07225738, -0.17472742, 0.1499059, -0.24601264, -0.25269535, 0.11262548, -0.08882318, 0.12241974, 0.35499957, -0.10845275, 0.013618386, 0.016696492, -0.069810525, -0.19361006, 0.33629563, -0.19772667, 0.011744207, -0.008974844, 0.0010091156, 0.22134513, -0.20966883, 0.03822157, 0.08804019, 0.11288315, 0.008376919, 0.15416574, 0.14673932, 0.04644713, -0.037493743, -0.07340382, 0.27585793, -0.13231725, 0.1780692, -0.40117204, -0.28636923, 0.2611396, 0.06258053, 0.21725261, 0.31685007, 0.124705434, -0.19767064, -0.42261192, 0.20958751, -0.022217626, -0.26786008, 0.037114482, -0.49277598, 0.32637754, 0.0078839995, 0.01650781, -0.14448854, 0.43707192, -0.06547795, -0.008750201, -0.013007096, 0.13033581, 0.34262726, -0.4561812, -0.14253624, -0.22308715, -0.105225846, 0.31123114, 0.08837291, 0.03347811, 0.011859631, 0.10455212, 0.19422622, 0.00193

### Ingested indexed collection of documents

In [89]:
make_request(
    f"_ingest/pipeline/rag-ingest-pipeline",
    "PUT", 
    {
        "description": "An RAG ingest pipeline",
        "processors": [{
            "text_embedding": {
                "model_id": model_id,
                "field_map": {
                    "text": "sentence_embedding"
                }
            }
        }]
    }
)

Status Code: 200
Response JSON: {'acknowledged': True}


{'acknowledged': True}

In [93]:
make_request(
    index_name,
    "PUT",
    {
        "settings": {
            "index.knn": "true",
            "default_pipeline": "rag-ingest-pipeline"
        },
        "mappings": {
            "properties": {
                "id": {"type": "text"},
                "sentence_embedding": {
                    "type": "knn_vector",
                    "dimension": 768,
                    "method": {
                        "name": "hnsw",
                        "engine": "faiss",
                        "space_type": "l2",
                        "parameters": {
                            "encoder": {
                                "name": "sq",
                                "parameters": {"type": "fp16"}
                            },
                            "ef_construction": 256,
                            "m": 8
                        }
                    }
                },
                "text": {"type": "text"}
            }
        }
    }
)

Status Code: 200
Response JSON: {'acknowledged': True, 'shards_acknowledged': True, 'index': 'knn-faiss-avx2-index3'}


{'acknowledged': True,
 'shards_acknowledged': True,
 'index': 'knn-faiss-avx2-index3'}

Import the dataset

In [94]:
from sklearn.datasets import fetch_20newsgroups

In [95]:
news = fetch_20newsgroups()

In [97]:
for idoc, doc in enumerate(news["data"][:100]):
    print(f"Adding doc #{idoc}")
    make_request(
        f"{index_name}/_doc",
        "POST",
        {
            "text": doc
        }
    )

Adding doc #0
Status Code: 201
Response JSON: {'_index': 'knn-faiss-avx2-index3', '_id': 'bu3TW54BnAiy9mbylsLC', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 2, 'failed': 0}, '_seq_no': 0, '_primary_term': 1}
Adding doc #1
Status Code: 201
Response JSON: {'_index': 'knn-faiss-avx2-index3', '_id': 'b-3TW54BnAiy9mbym8KP', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 2, 'failed': 0}, '_seq_no': 1, '_primary_term': 1}
Adding doc #2
Status Code: 201
Response JSON: {'_index': 'knn-faiss-avx2-index3', '_id': 'cO3TW54BnAiy9mbyp8If', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 2, 'failed': 0}, '_seq_no': 2, '_primary_term': 1}
Adding doc #3
Status Code: 201
Response JSON: {'_index': 'knn-faiss-avx2-index3', '_id': 'ce3TW54BnAiy9mbyrMJu', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 2, 'failed': 0}, '_seq_no': 3, '_primary_term': 1}
Adding doc #4
Status Code: 201
Response JSON: {'

Test a query

In [99]:
make_request(
    f"{index_name}/_search",
    "GET",
    {
        "_source": {
            "excludes": [
                "sentence_embedding"
            ]
        },
        "query": {
            "neural": {
                "sentence_embedding": {
                    "query_text": "What is AutoDoubler?",
                    "model_id": model_id,
                    "k": 5
                }
            }
        }
    }
)

Status Code: 200
Response JSON: {'took': 685, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 5, 'relation': 'eq'}, 'max_score': 0.0144805405, 'hits': [{'_index': 'knn-faiss-avx2-index3', '_id': 'bu3TW54BnAiy9mbylsLC', '_score': 0.0144805405, '_source': {'text': "From: lerxst@wam.umd.edu (where's my thing)\nSubject: WHAT car is this!?\nNntp-Posting-Host: rac3.wam.umd.edu\nOrganization: University of Maryland, College Park\nLines: 15\n\n I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a Bricklin. The doors were really small. In addition,\nthe front bumper was separate from the rest of the body. This is \nall I know. If anyone can tellme a model name, engine specs, years\nof production, where this car is made, history, or whatever info you\nhave on this funky looking car, please e-mail.\n\nThank

{'took': 685,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 5, 'relation': 'eq'},
  'max_score': 0.0144805405,
  'hits': [{'_index': 'knn-faiss-avx2-index3',
    '_id': 'bu3TW54BnAiy9mbylsLC',
    '_score': 0.0144805405,
    '_source': {'text': "From: lerxst@wam.umd.edu (where's my thing)\nSubject: WHAT car is this!?\nNntp-Posting-Host: rac3.wam.umd.edu\nOrganization: University of Maryland, College Park\nLines: 15\n\n I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a Bricklin. The doors were really small. In addition,\nthe front bumper was separate from the rest of the body. This is \nall I know. If anyone can tellme a model name, engine specs, years\nof production, where this car is made, history, or whatever info you\nhave on this funky looking car, please e-mail.\n\nThanks,\n- IL\n   

## Integrate the LLM

First, define the model and allow connection

In [114]:
model_endpoint="10.152.183.95/openai"
model_name="qwen25-3b-native"

In [101]:
make_request(
    "_cluster/settings", 
    "PUT",
    {
        "persistent": {
            "plugins.ml_commons.connector.private_ip_enabled": True,
            "plugins.ml_commons.trusted_connector_endpoints_regex": [
                "^http://.*$"
            ]
        }
    }
)

Status Code: 200
Response JSON: {'acknowledged': True, 'persistent': {'plugins': {'ml_commons': {'connector': {'private_ip_enabled': 'true'}, 'trusted_connector_endpoints_regex': ['^http://.*$']}}}, 'transient': {}}


{'acknowledged': True,
 'persistent': {'plugins': {'ml_commons': {'connector': {'private_ip_enabled': 'true'},
    'trusted_connector_endpoints_regex': ['^http://.*$']}}},
 'transient': {}}

### Create the connections to the external LLM

In [102]:
connector_id = make_request(
    "_plugins/_ml/connectors/_create",
    "POST",
    {
        "name": "vLLM Connector",
        "description": "OpenAI-compatible vLLM API",
        "version": 2,
        "protocol": "http",
        "parameters": {
            "endpoint": model_endpoint,
            "model": model_name,
            "temperature": 0.03,
            "repetition_penalty": 1.13
        },
        "credential": {
            "api_key": "not-needed-for-local-vllm"
        },
        "actions": [{
            "action_type": "predict",
            "method": "POST",
            "url": "http://${parameters.endpoint}/v1/chat/completions",
            "request_body": "{ \"model\": \"${parameters.model}\", \"messages\": ${parameters.prompt}, \"temperature\":${parameters.temperature}, \"repetition_penalty\": ${parameters.repetition_penalty} }"
        }]
    }
)["connector_id"]

Status Code: 200
Response JSON: {'connector_id': 'tO3aW54BnAiy9mbyQMKY'}


In [103]:
task_id_llm = make_request(
    "_plugins/_ml/models/_register",
    "POST",
    {
        "name": f"KServe Inference Service: {model_name}",
        "function_name": "remote",
        "description": "RAG LLM test model",
        "connector_id": connector_id
    }
)["task_id"]

Status Code: 200
Response JSON: {'task_id': 'vO3bW54BnAiy9mbydsKb', 'status': 'CREATED', 'model_id': 've3bW54BnAiy9mbydsLi'}


In [104]:
model_id_llm = get_model_id(task_id_llm)

Status Code: 200
Response JSON: {'model_id': 've3bW54BnAiy9mbydsLi', 'task_type': 'REGISTER_MODEL', 'function_name': 'REMOTE', 'state': 'COMPLETED', 'worker_node': ['Pxs3SECyTqKo3771gblWrg'], 'create_time': 1779657569945, 'last_update_time': 1779657570071, 'is_async': False}


In [105]:
model_id_llm

've3bW54BnAiy9mbydsLi'

In [106]:
make_request(
    f"_plugins/_ml/models/{model_id_llm}/_deploy",
    "POST",
    {}
)

Status Code: 200
Response JSON: {'task_id': 'vu3cW54BnAiy9mbyysKA', 'task_type': 'DEPLOY_MODEL', 'status': 'COMPLETED'}


{'task_id': 'vu3cW54BnAiy9mbyysKA',
 'task_type': 'DEPLOY_MODEL',
 'status': 'COMPLETED'}

### Define Conversational Agent

In [107]:
agent_id = make_request(
    "_plugins/_ml/agents/_register",
    "POST",
    {
        "name": "test-agent-vector-mlmodel-rag",
        "type": "conversational_flow",
        "description": "This is a demo agent for conversational flow",
        "app_type": "rag",
        "memory": {
            "type": "conversation_index"
        },
        "tools": [
            {
                "type": "VectorDBTool",
                "parameters": {
                    "model_id": model_id,
                    "index": "knn-faiss-avx2-index1",
                    "embedding_field": "sentence_embedding",
                    "source_field": ["text"],
                    "input": "${parameters.question}",
                    "doc_size": 10,
                    "k": 30
                }
            },
            {
                "type": "MLModelTool",
                "description": "A RAG LLM tool to answer questions",
                "parameters": {
                    "model_id": model_id_llm,
                    "prompt": "[ { \"role\": \"Context:\", \"content\": \"${parameters.VectorDBTool.output}\" }, { \"role\": \"Chat history:\", \"content\": \"${parameters.chat_history:-}\" }, { \"role\": \"User instruction:\", \"content\":\"${parameters.question}\" } ]"
                }
            }
        ]
    }
)["agent_id"]

Status Code: 200
Response JSON: {'agent_id': 'v-3dW54BnAiy9mbyr8JZ'}


In [108]:
agent_id

'v-3dW54BnAiy9mbyr8JZ'

Test out the agent

In [109]:
inference_output = make_request(
    f"_plugins/_ml/agents/{agent_id}/_execute",
    "POST",
    {
        "parameters": {
            "question": "What is the best Linux distribution?"
        }
    }
)

Status Code: 200
Response JSON: {'inference_results': [{'output': [{'name': 'memory_id', 'result': 'hKbeW54B5NKlTbeIyHWE'}, {'name': 'parent_message_id', 'result': 'habeW54B5NKlTbeIyHXT'}, {'name': 'MLModelTool', 'result': '{"id":"chatcmpl-af4b6dde1a142ce2","object":"chat.completion","created":1.779657789E9,"model":"qwen25-3b-native","choices":[{"index":0.0,"message":{"role":"assistant","content":"Hello! I\\u0027m Qwen, an artificial intelligence designed to assist with various tasks and provide information on a wide range of topics. How can I help you today? Whether it\\u0027s answering questions, providing explanations, or helping with calculations, feel free to ask me anything within my capabilities.","tool_calls":[]},"finish_reason":"stop"}],"usage":{"prompt_tokens":24.0,"total_tokens":82.0,"completion_tokens":58.0}}'}]}]}


In [110]:
memory_id = [output["result"] for output in inference_output["inference_results"][0]["output"] if output["name"]=="memory_id"][0]

In [111]:
memory_id

'hKbeW54B5NKlTbeIyHWE'

In [112]:
inference_output = make_request(
    f"_plugins/_ml/agents/{agent_id}/_execute",
    "POST",
    {
        "parameters": {
            "question": "Find a document about AutoDoubler.",
            "memory_id": memory_id,
            "message_history_limit": 10
        }
    }
)

Status Code: 200
Response JSON: {'inference_results': [{'output': [{'name': 'memory_id', 'result': 'hKbeW54B5NKlTbeIyHWE'}, {'name': 'parent_message_id', 'result': 'OqPfW54B8MAT2xzPerbu'}, {'name': 'MLModelTool', 'result': '{"id":"chatcmpl-b710aa9c3bfa67d9","object":"chat.completion","created":1.779657833E9,"model":"qwen25-3b-native","choices":[{"index":0.0,"message":{"role":"assistant","content":"Hello! I\\u0027m Qwen, an artificial intelligence designed to assist with various tasks and provide information on a wide range of topics. How can I help you today? Whether it\\u0027s answering questions, providing explanations, or helping with calculations, feel free to ask me anything within my capabilities.","tool_calls":[]},"finish_reason":"stop"}],"usage":{"prompt_tokens":24.0,"total_tokens":82.0,"completion_tokens":58.0}}'}]}]}
